# Fairlearn: Bias Detection and Mitigation
### Azure AI Engineer Training Series

> This notebook is a **read-only reference guide**. It introduces the concepts and tools involved in using Fairlearn within an Azure AI environment. No code is executed here. Use this as your reference while working through the hands-on exercises.

## 1. What is Fairlearn?

Fairlearn is an **open-source Python toolkit from Microsoft** that empowers data scientists and developers to assess and improve the fairness of their AI systems.

It has two main components:

| Component | What it does |
|---|---|
| **Interactive dashboard** | Visualise how a model impacts different groups; compare multiple models on fairness and performance |
| **Mitigation algorithms** | Reduce unfairness in classification and regression models using postprocessing or reduction techniques |

### Why it matters

AI systems can behave unfairly for reasons that are both **societal** (biased training data, historical inequalities) and **technical** (too few data points for some groups, proxy variables). Because these sources of unfairness are complex and overlapping, it is not possible to fully eliminate bias — the goal is to **mitigate fairness-related harms as much as possible**.

Fairlearn focuses specifically on two types of harm:

| Harm Type | Description | Example |
|---|---|---|
| **Allocation harm** | The model unfairly withholds opportunities, resources, or information from a group | A resume-screening model trained on historically biased hiring data withholds job opportunities from women |
| **Quality-of-service harm** | The model provides a lower quality of service to some groups than others | A facial recognition system performs less accurately for users with certain skin tones |

## 2. Lab Scenario

A financial services company uses an ML model to predict whether a loan applicant earns more than **$50,000 per year**. The output informs — but does not replace — a human credit officer's decision.

Regulators have requested a fairness audit confirming the model does not discriminate on the basis of **sex**.

**Dataset:** UCI Adult Income — 48,842 rows of US census data  
**Target:** Binary — income above or below $50,000  
**Sensitive feature:** `sex` (Male / Female)

### What the raw data shows

Before any model is trained, the dataset already contains a significant income gap between groups:

| Group | % earning > $50K |
|---|---|
| Male | ~31% |
| Female | ~11% |

> This gap exists in the data because it reflects historical societal inequality. A model trained on this data will **learn and perpetuate** this disparity unless actively mitigated. This is why post-training fairness assessment is essential — even when the sensitive attribute is excluded from model inputs.

## 3. Fairness Metrics

There is no single universal definition of fairness. Fairlearn lets you choose the metric that fits your legal and ethical context. The four main metrics are:

| Metric | What it requires | Best used for |
|---|---|---|
| **Demographic Parity** | Equal positive prediction rate across all groups | Hiring, advertising, credit pre-screening |
| **Equalized Odds** | Equal true positive rate AND false positive rate across groups | Medical diagnosis, recidivism scoring |
| **Equal Opportunity** | Equal true positive rate (recall) across groups | College admissions, parole decisions |
| **Bounded Group Loss** | Maximum loss difference between groups below a threshold | Regression tasks, ranking systems |

### Measuring disparity with MetricFrame

`MetricFrame` is the core assessment object in Fairlearn. It computes any set of metrics both **overall** and **per group**, then exposes disparity through two methods:

| Method | What it returns | Perfect parity value |
|---|---|---|
| `.difference()` | Maximum metric value minus minimum across all groups | `0` |
| `.ratio()` | Minimum metric value divided by maximum across all groups | `1.0` |

**Intersectional analysis:** MetricFrame also supports passing multiple sensitive features simultaneously (e.g. sex and race combined). This often reveals compounded disadvantages that are hidden when features are assessed independently.

## 4. Mitigation Algorithms

Fairlearn includes two types of mitigation algorithms, both of which wrap any standard scikit-learn estimator:

| Type | How it works | Needs sensitive feature at deployment? |
|---|---|---|
| **Postprocessing** | Adjusts thresholds on an already-trained model's predictions to satisfy a fairness constraint — no retraining required | Yes |
| **Reductions** | Iteratively re-weights training data and retrains the model until the fairness constraint is satisfied | No |

### ExponentiatedGradient

A **reduction algorithm** and the recommended default for binary classification. It finds a randomised classifier that minimises overall loss subject to your chosen fairness constraint.

Key parameters:

| Parameter | Description | Typical value |
|---|---|---|
| `estimator` | Any scikit-learn classifier | `LogisticRegression()` |
| `constraints` | Fairness constraint instance | `DemographicParity()` |
| `eps` | Maximum allowed constraint violation | `0.01` to `0.05` |
| `max_iter` | Maximum number of iterations | `50` |

### GridSearch

Trains a **family of models** across a grid of fairness constraint weights, returning all models on the accuracy–fairness Pareto frontier. Use this when you want to present stakeholders with a range of options rather than a single model.

## 5. Lab Steps Overview

This lab walks you through a complete fairness audit — from loading data to deploying a mitigated model. Here is a plain summary of what happens at each stage.

**Load the data**
You start by loading the UCI Adult Income dataset, which contains census information for around 48,000 people. The goal is to predict whether someone earns above $50,000 a year. The `sex` column is set aside as the sensitive feature — it is not fed into the model, but it is used later to check whether the model's predictions are fair across groups.

**Train a baseline model**
A basic logistic regression model is trained using five numeric columns: age, years of education, hours worked per week, capital gain, and capital loss. The model achieves around 82–84% overall accuracy. That sounds reasonable — but overall accuracy hides whether certain groups are being treated differently, which is what the next step reveals.

**Measure fairness with MetricFrame**
MetricFrame breaks down model performance by group. Instead of one overall accuracy number, you get a separate number for Male applicants and Female applicants across metrics like selection rate, true positive rate, and false positive rate. The disparity between groups turns out to be significant — roughly 18 to 22 percentage points difference in selection rate, which is well above the 10-point threshold commonly used in financial services regulation.

**Visualise the disparity**
The same group-level numbers are plotted as bar charts so the gap is immediately visible. A dashed line shows the overall metric value. Any bar sitting well above or below that line tells you one group is receiving meaningfully different outcomes from the model. This chart is what you would show in a governance review.

**Apply mitigation**
The `ExponentiatedGradient` algorithm is applied with a `DemographicParity` constraint. It iteratively adjusts the training process until the selection rate difference between groups drops below the allowed threshold of 2%. After mitigation, the disparity falls from around 0.20 down to under 0.02.

**Compare the models**
A side-by-side table and scatter chart show the baseline and mitigated model together. The mitigated model is fairer — lower disparity — but slightly less accurate. That small accuracy cost is the trade-off. Stakeholders need to see this chart and agree on the deployment decision before any model goes to production.

**Register and deploy**
The mitigated model is saved, tagged with its fairness metadata, and registered in Azure ML. It is then deployed as a live REST endpoint on Azure AI infrastructure so it can receive real inference requests.

## 6. Registering and Deploying to an Azure AI Endpoint

Once the mitigated model has been selected, it needs to be made available for use. In Azure, this is done by registering the model in the Azure ML model registry and then deploying it as a **Managed Online Endpoint** — a REST API that other applications can call to get predictions.

**Model registration**
The model is saved as a `.pkl` file and uploaded to the Azure ML workspace. During registration, a set of tags are attached to it — things like which mitigation algorithm was used, which fairness constraint was applied, and which sensitive feature was audited. These tags make it possible for a governance or compliance team to search, filter, and review models in Azure ML Studio without needing to re-run any code.

**The endpoint**
A Managed Online Endpoint is a named URL hosted and managed by Azure. Once the model is deployed to it, any authorised application can send a JSON request and receive a prediction back. Authentication is handled by an API key or an Azure AD token — no credentials are stored in the code.

**The scoring script**
The deployment requires a small Python file called `score.py`. It has two jobs: load the model when the endpoint starts up, and handle each incoming prediction request. Azure calls these two functions automatically — the engineer just needs to make sure the input data is parsed correctly and the model's output is returned in the expected format.

**Verification**
After deployment, the endpoint can be tested directly from Azure ML Studio under the **Endpoints** tab, or by sending a request from any HTTP client. The Responsible AI dashboard in Azure ML Studio also lets you view the fairness tags, compare model variants, and export a PDF governance report suitable for a regulatory submission.

**Credentials and security**
All workspace details — subscription ID, resource group, workspace name, and endpoint URL — are loaded from environment variables, never written into the notebook. The `DefaultAzureCredential` class automatically picks up whichever Azure identity is available: an `az login` session during local development, or a Managed Identity when running inside Azure. This means the same code works in both environments without any changes.

## 7. Viewing Results in Azure ML Studio

After completing the lab steps, verify your work in the Azure ML Studio UI:

**1. Check the registered model**  
Go to `ml.azure.com` → your workspace → **Models** → `fairlearn-income-classifier v1`. Click the model to view the fairness tags you attached during registration.

**2. Open the Responsible AI dashboard**  
Click the **Responsible AI** tab on the model detail page. Select `sex` as the sensitive feature and `selection_rate` as the metric. The same disparity bars from your notebook now appear as an interactive chart.

**3. Compare models**  
Use the **Model comparison** view to load both the baseline and mitigated model predictions side by side. This view is useful for stakeholder presentations.

**4. Check the live endpoint**  
Go to **Endpoints** → `fairlearn-income-endpoint` → **Consume** tab to find the scoring URI and test the endpoint directly from the Studio UI.

**5. Export a governance report**  
Click **Export PDF** in the Responsible AI dashboard to download a report suitable for a regulatory submission or internal audit.

## 8. Key Takeaways

**Fairness must be measured, not assumed**  
High overall accuracy does not mean the model treats all groups equitably. Always run `MetricFrame` before any production deployment and make it a mandatory step in your evaluation pipeline.

**There is no single definition of fairness**  
`DemographicParity`, `EqualizedOdds`, and `EqualOpportunity` encode different ethical and legal positions. The right choice depends on the regulatory context and the relative consequences of false positives versus false negatives in your application.

**Mitigation involves a trade-off**  
`ExponentiatedGradient` reduces disparity at a small accuracy cost. Quantify this trade-off, present it clearly to stakeholders, and get documented sign-off before deployment.

**Excluding the sensitive feature is not enough**  
Even when `sex` or `race` is excluded from model inputs, proxy variables such as `occupation` or `marital-status` can carry the same information indirectly. Post-training fairness assessment with `MetricFrame` is still required.

**Azure ML is the governance backbone**  
Model registration with fairness tags, the Responsible AI dashboard, and Managed Online Endpoints together create the versioned, auditable record required by frameworks including the EU AI Act and Microsoft's Responsible AI Standard.



## Further Reading

| Resource | Link |
|---|---|
| Fairlearn documentation | https://fairlearn.org |
| Bird et al. (2020) whitepaper | https://www.microsoft.com/en-us/research/publication/fairlearn-a-toolkit-for-assessing-and-improving-fairness-in-ai/ |
| Azure ML Responsible AI dashboard | https://aka.ms/rai-dashboard |
| Microsoft Responsible AI Standard | https://www.microsoft.com/en-us/ai/responsible-ai |
| Azure AI Engineer certification (AI-102) | https://learn.microsoft.com/en-us/certifications/azure-ai-engineer/ |